# Change in mean

The most common change-detection problem is detecting shifts in the mean of a time series. Skchange offers several scorers for this feature; the two most useful entry points are `L2Cost` paired with `PELT` for exact optimisation, and `CUSUM` paired with `SeededBinarySegmentation` or `MovingWindow` for a faster approximate search.

In [ ]:
import plotly.io as pio

from skchange.datasets import generate_piecewise_normal_data
from skchange.utils.plotting import plot_detections

pio.renderers.default = "notebook"

X = generate_piecewise_normal_data(
    means=[0, 10, 0, -3, 5, 1],
    lengths=[30, 5, 15, 50, 60, 40],
    seed=0,
)

The series has six segments of unit-variance Gaussian data with five underlying changepoints at indices 30, 35, 50, 100, and 160. The second segment is a short spike that is easy to miss with a coarse search.

## `PELT` and `L2Cost`

`PELT` minimises the total sum of segment costs plus a per-changepoint penalty. With `L2Cost` — the sum of squared deviations from each segment's sample mean — this recovers all five changepoints on the shared `X`.

In [ ]:
from skchange.detectors import PELT
from skchange.interval_scorers import L2Cost

detector = PELT(L2Cost(), penalty=10.0)
changepoints = detector.fit_predict(X)

plot_detections(X, changepoints=changepoints).show()
print(changepoints)

## `FPOP` for univariate data
If you have a long series of univariate data and are looking for changes in the mean, `FPOP` is your best option.
It solves the same optimisation problem as `PELT`, but is more than 1000 times faster with our implementation.
It only works for univariate data though, and is so far restricted to the L2 cost function.

In [ ]:
from skchange.detectors import FPOP

detector = FPOP(penalty=10.0)
changepoints = detector.fit_predict(X)

plot_detections(X, changepoints=changepoints).show()
print(changepoints)

## `SeededBinarySegmentation` and `CUSUM`

For long series, an exact `PELT` search can become expensive. `SeededBinarySegmentation` evaluates a change score on a pre-computed grid of intervals and picks the local maxima that exceed the penalty. Paired with the classical `CUSUM` statistic for a change in mean, it recovers the same changepoints at a fraction of the time.

In [ ]:
from skchange.detectors import SeededBinarySegmentation
from skchange.interval_scorers import CUSUM

detector = SeededBinarySegmentation(CUSUM(), min_subinterval_length=2, penalty=5.0)
changepoints = detector.fit_predict(X)

plot_detections(X, changepoints=changepoints).show()
print(changepoints)

Note the `min_subinterval_length=2`: with the default of 5, the two-sample-wide spike segment `[30, 35)` would fall through the grid.

## `MovingWindow` and `CUSUM`

`MovingWindow` slides one or more fixed-width windows across the series and evaluates a change score at the window's midpoint. The length from the midpoint to the boundaries of the window is called the *bandwidth*. Beyond `predict`, `MovingWindow` exposes `predict_scores`, which returns the penalised score at every candidate changepoint. This is a useful visual diagnostic that shows how much evidence there is for a change at each location and how sensitive that evidence is to the bandwidth.

In [ ]:
import pandas as pd
import plotly.express as px

from skchange.detectors import MovingWindow

detector = MovingWindow(CUSUM(), bandwidth=[5, 15, 40], penalty_scale=1.0).fit(X)
changepoints = detector.predict(X)
scores, index = detector.predict_scores(X, return_index=True)

score_df = pd.DataFrame(
    {
        "split": index["splits"],
        "bandwidth": index["bws"].astype(str),
        "penalised_score": scores,
    }
)
fig = px.line(
    score_df,
    x="split",
    y="penalised_score",
    color="bandwidth",
    labels={"split": "Candidate changepoint", "penalised_score": "Penalised score"},
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
for cp in changepoints:
    fig.add_vline(x=int(cp), line_dash="dot", line_color="red")
fig.show()
print(changepoints)

Peaks above the dashed zero line mark candidate changepoints; `MovingWindow` keeps the local maxima. Small bandwidths react to short segments like the `[30, 35)` spike, while larger bandwidths smooth over noise and pinpoint sustained shifts more sharply. Comparing curves across bandwidths often reveals whether a detection is well-supported at multiple scales or only picked up at one.

## Choosing a detector

The choice of detector comes down to the usual trade-offs between exact optimisation and fast approximate search. See the [Detectors](../detectors/index.rst) section for a side-by-side comparison and per-detector notes. For guidance on setting the penalty, see the [Penalties calibration page](../tuning/penalty_calibration.ipynb).